In [4]:
!pip -q install gradio requests

import requests
import gradio as gr

API_KEY = "05ffffb1a6b14887b143d20a6defb5da"


def get_city_news(city):
    if not city.strip():
        return "⚠️ Please enter a city name."

    city = city.strip()

    # NewsAPI search URL
    url = "https://newsapi.org/v2/everything"

    params = {
        "q": city,
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": 10,
        "apiKey": API_KEY
    }

    try:
        response = requests.get(url, params=params, timeout=10)

        if response.status_code != 200:
            return f"❌ NewsAPI Error: {response.status_code}\n\n{response.text}"

        data = response.json()

        if data.get("status") != "ok":
            return f"❌ {data.get('message', 'Unable to fetch news.')}"

        articles = data.get("articles", [])

        if not articles:
            return f"🔎 No recent news found for **{city}**."

        output = f"# 📰 Latest News for {city}\n\n"

        for i, article in enumerate(articles, start=1):

            title = article.get("title", "No title")
            description = article.get("description", "No description available.")
            source = article.get("source", {}).get("name", "Unknown source")
            published = article.get("publishedAt", "")
            article_url = article.get("url", "#")

            output += f"""
### {i}. {title}

**Source:** {source}
**Published:** {published}

{description}

🔗 [Read Full Article]({article_url})

---

"""

        return output

    except requests.exceptions.RequestException as e:
        return f"❌ Connection error: {str(e)}"


# -----------------------------
# Gradio Dashboard
# -----------------------------

with gr.Blocks(title="City News Dashboard") as dashboard:

    gr.Markdown(
        """
        # 📰 City News Dashboard
        ### Search the latest news by city
        Enter a city name below to find recent news related to that city.
        """
    )

    with gr.Row():

        city_input = gr.Textbox(
            label="City Name",
            placeholder="e.g. Lahore, London, New York",
            scale=4
        )

        search_button = gr.Button(
            "🔍 Get News",
            variant="primary",
            scale=1
        )

    news_output = gr.Markdown(
        value="Enter a city and click **Get News**."
    )

    search_button.click(
        fn=get_city_news,
        inputs=city_input,
        outputs=news_output
    )

    city_input.submit(
        fn=get_city_news,
        inputs=city_input,
        outputs=news_output
    )


dashboard.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ebd680959711f52058.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
